In [0]:
%pip install seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [0]:
df = spark.table('flight_delay_predictor_project.default.first_prediction_try_1').toPandas()

In [0]:
from sklearn.model_selection import train_test_split
df["weather_delay"] = df["weather_delay"].fillna(0)

In [0]:
from sklearn.tree import DecisionTreeClassifier
dtree = DecisionTreeClassifier()

In [0]:
df = spark.createDataFrame(df)

In [0]:
from pyspark.sql import Window, functions as F

hr_delay = (df.withColumn("delay_in_hr", 
F.when(F.col("weather_delay") < 1, "no delay")
.when(F.col("weather_delay") < 60, "less than 1") 
.when(F.col("weather_delay") < 120, "1+")
.when(F.col("weather_delay") < 180, "2+")
.when(F.col("weather_delay") < 240, "3+")
.when(F.col("weather_delay") < 300, "4+")
.when(F.col("weather_delay") < 360, "5+")
.when(F.col("weather_delay") < 420, "6+")
.when(F.col("weather_delay") < 480, "7+")
.when(F.col("weather_delay") < 540, "8+")
.when(F.col("weather_delay") < 600, "9+")
.otherwise("10+")) 
.select("temperature_2m", "relative_humidity_2m", "precipitation", "snowfall", "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m", "cloud_cover", "pressure_msl", "weather_code", "delay_in_hr"))

In [0]:
hr_delay = hr_delay.toPandas()
print(hr_delay)

In [0]:
flights_1 = hr_delay[hr_delay['delay_in_hr'] != 'no delay']
flights_2 = hr_delay[hr_delay['delay_in_hr'] == 'no delay']
print(flights_1.shape)
print(flights_2.shape)

In [0]:
from sklearn.utils import resample
flights_2_downsampled = resample(flights_2,random_state=42,n_samples=845,replace=False)
flights_downsampled = pd.concat([flights_2_downsampled,flights_1])

In [0]:
X = flights_downsampled.drop('delay_in_hr',axis=1)
y = flights_downsampled['delay_in_hr']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30,random_state=42)
dtree = DecisionTreeClassifier()

In [0]:
dtree.fit(X_train,y_train)

In [0]:
from sklearn.metrics import classification_report,confusion_matrix
predictions = dtree.predict(X_test)
print(classification_report(y_test,predictions))
print(confusion_matrix(y_test,predictions))

In [0]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, predictions)